<a href="https://colab.research.google.com/github/NaghamZidiah/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip -q install duckdb

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of analysis

One row represents the daily performance of one content page for one client on one report date.

This notebook uses the `fact_content_daily_performance` table.

For exploration, I will use a mid-panel month (2026-03) instead of the latest month, following the internship recommendation to avoid developing logic on the final outcome window.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

These observed performance metrics will be used as input features to describe content behavior.

### Label

This project uses clustering, so there is no predefined target label.

### Context
- report_date
- client_hash_id
- content_hash_id

These fields provide context and identify each observation but are not used as predictive features.

### Excluded
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other

These AI traffic source columns are excluded from the initial clustering to keep the first analysis focused on core search and engagement signals.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The following queries verify the main claims of the data contract for the March 2026 data slice.

### Query 1 — Row count and date range

This query verifies the selected time window (March 2026) and shows the number of observations available.

In [ ]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded successfully!")

Token loaded successfully!


In [ ]:
con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("Connected successfully!")

Connected successfully!


In [ ]:
rel = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


### Query 2 — Verify the grain

This query checks whether each combination of report date, client, and content appears only once.

In [ ]:
query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicates
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicates


### Query 3 — Data availability

This query verifies how many rows have Google Search Console data available using the required IS TRUE condition.

In [ ]:
query = f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


### Verification summary

The queries confirm that:

- The selected data covers March 2026.
- Each row represents one report date, one client, and one content item.
- The selected March 2026 slice contains more than three million rows with Google Search Console data available, based on the availability flag.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


This dataset shows observed content performance patterns, but it cannot explain the exact reasons behind changes.

- It does not prove causation, only relationships between metrics.
- Some content may have limited history, so long-term trends may not be complete.
- Missing GSC data availability can limit parts of the analysis.
- Results should be used for decision support and pattern discovery, not as guaranteed predictions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.